# Part 2: Load cached model + datasets and run evals on 2 GPUs

In [1]:
!pip install -U unsloth transformers datasets huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 28.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 54.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 76.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.4/418.4 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.7 MB/s eta 0:00:00:00:01
 

In [ ]:
%%writefile /kaggle/working/run_eval_single.py
import os
import re
import json
import time
import argparse
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from unsloth import FastLanguageModel
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# ----------------------------
# Args
# ----------------------------
parser = argparse.ArgumentParser()
parser.add_argument("--mode", type=str, required=True, choices=["english", "language"])
parser.add_argument("--batch_size", type=int, default=128)
parser.add_argument("--n_samples", type=int, default=None)
args = parser.parse_args()

# ----------------------------
# Env / config
# ----------------------------
os.environ["KAGGLE_DATASET_INPUT_DIR"] = "/kaggle/input/datasets/rashikshahjahan/ayaeval2/eval_unsloth_artifacts/datasets"

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

cond = "condition-2-ur-5k"
ADAPTER_REPO = "legesher/language-decoded-lora-condition-2-ur-5k"
MAX_SEQ_LENGTH = 1024
DTYPE = torch.float16
LOAD_IN_4BIT = True

DATASET_INPUT_DIR = Path(os.environ.get("KAGGLE_DATASET_INPUT_DIR"))

# ----------------------------
# HF auth
# ----------------------------
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("HF_TOKEN")
login(token=token)

# ----------------------------
# Helpers
# ----------------------------


def load_model():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required.")

    # Because CUDA_VISIBLE_DEVICES is set per process, this process sees only one GPU.
    # So we always bind to cuda:0 inside the process.
    torch.cuda.set_device(0)

    print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
    print("torch.cuda.device_count() =", torch.cuda.device_count())
    print("current_device =", torch.cuda.current_device())
    print("device_name =", torch.cuda.get_device_name(0))

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=ADAPTER_REPO,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT,
        token=token,
        device_map={"": 0},
    )

    FastLanguageModel.for_inference(model)
    model.eval()

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    return model, tokenizer

def load_cached_dataframe(name: str):
    path = DATASET_INPUT_DIR / f"{name}.jsonl"
    if not path.exists():
        raise FileNotFoundError(f"Missing dataset cache: {path}")
    return pd.read_json(path, lines=True)

def load_all_datasets():
    return {
        "mgsm_zh": load_cached_dataframe("mgsm_zh"),
        "mgsm_es": load_cached_dataframe("mgsm_es"),
        "mgsm_ur": load_cached_dataframe("mgsm_ur"),
        "xnli_zh": load_cached_dataframe("xnli_zh"),
        "xnli_es": load_cached_dataframe("xnli_es"),
        "xnli_ur": load_cached_dataframe("xnli_ur"),
        "csqa_zh": load_cached_dataframe("csqa_zh"),
        "csqa_es": load_cached_dataframe("csqa_es"),
        "csqa_ur": load_cached_dataframe("csqa_ur"),
    }

def prompt_suffix_for_lang(lang: str) -> str:
    return "english" if lang == "en" else "language"

def get_tokenized_prompts(eval_df, lang: str, benchmark: str):
    prompt_suffix = prompt_suffix_for_lang(lang)
    input_ids_column = f"input_ids_{prompt_suffix}"
    attention_mask_column = f"attention_mask_{prompt_suffix}"
    missing_columns = [
        column
        for column in [input_ids_column, attention_mask_column]
        if column not in eval_df.columns
    ]
    if missing_columns:
        raise KeyError(
            f"Missing {missing_columns} for {benchmark}; rerun the preprocessing notebook "
            "to regenerate cached datasets with tokenized prompts."
        )
    return list(zip(eval_df[input_ids_column].tolist(), eval_df[attention_mask_column].tolist()))

def pad_tokenized_batch(batch, tokenizer):
    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    max_length = max(len(input_ids) for input_ids, _ in batch)
    padded_input_ids = []
    padded_attention_mask = []

    for input_ids, attention_mask in batch:
        input_ids = list(input_ids)
        attention_mask = list(attention_mask)
        pad_length = max_length - len(input_ids)
        if tokenizer.padding_side == "left":
            padded_input_ids.append([pad_token_id] * pad_length + input_ids)
            padded_attention_mask.append([0] * pad_length + attention_mask)
        else:
            padded_input_ids.append(input_ids + [pad_token_id] * pad_length)
            padded_attention_mask.append(attention_mask + [0] * pad_length)

    return {
        "input_ids": torch.tensor(padded_input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(padded_attention_mask, dtype=torch.long),
    }

def generate_texts_batch(
    model,
    tokenizer,
    tokenized_prompts,
    max_new_tokens=80,
    batch_size=32,
    desc="",
):
    all_outputs = []
    model_device = next(model.parameters()).device

    for i in tqdm(range(0, len(tokenized_prompts), batch_size), desc=desc or "Generating", unit="batch"):
        batch = tokenized_prompts[i : i + batch_size]
        inputs = pad_tokenized_batch(batch, tokenizer)
        prompt_length = inputs["input_ids"].shape[1]
        inputs = {key: value.to(model_device) for key, value in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        for output in outputs:
            text = tokenizer.decode(output[prompt_length:], skip_special_tokens=True).strip()
            all_outputs.append(text)

    return all_outputs

# ----------------------------
# MGSM
# ----------------------------
# Prompts, prompt tokens, and gold normalization are created in preprocessing.
NUMBER_RE = re.compile(r"-?\d+(?:\.\d+)?")

def extract_number(text: str):
    matches = NUMBER_RE.findall(text.replace(",", ""))
    return matches[-1] if matches else None

def evaluate_mgsm(model, tokenizer, df, lang: str, n_samples=None, batch_size: int = 32):
    eval_df = df if n_samples is None else df.head(n_samples)
    tokenized_prompts = get_tokenized_prompts(eval_df, lang, "MGSM")
    outputs = generate_texts_batch(model, tokenizer, tokenized_prompts, max_new_tokens=80, batch_size=batch_size, desc=f"MGSM-{lang}")

    rows, correct = [], 0
    for (_, row), output in zip(eval_df.iterrows(), outputs):
        pred = extract_number(output)
        gold = row["gold"]
        is_correct = pred == gold
        correct += int(is_correct)
        rows.append({
            "question": row["Question"],
            "raw_output": output,
            "pred": pred,
            "gold": gold,
            "correct": is_correct
        })
    return correct / len(rows) if rows else 0.0, pd.DataFrame(rows)

# ----------------------------
# XNLI
# ----------------------------
NATIVE_LABEL_MAP = {
    "蕴含": "entailment", "蕴涵": "entailment",
    "矛盾": "contradiction",
    "中立": "neutral",
    "implicación": "entailment", "implicacion": "entailment",
    "contradicción": "contradiction", "contradiccion": "contradiction",
    "لازمی": "entailment",
    "تردید": "contradiction",
    "غیرجانبدار": "neutral",
}

# Prompts, prompt tokens, and gold normalization are created in preprocessing.
XNLI_LABEL_RES = {
    label: re.compile(rf"\b{label}\b")
    for label in ["entailment", "contradiction", "neutral"]
}

def extract_xnli_label(text: str):
    text_lower = text.strip().lower()
    for label, label_re in XNLI_LABEL_RES.items():
        if label_re.search(text_lower):
            return label
    for native, english in NATIVE_LABEL_MAP.items():
        if native in text.strip():
            return english
    return None

def evaluate_xnli(model, tokenizer, df, lang: str, n_samples=None, batch_size: int = 32):
    eval_df = df if n_samples is None else df.head(n_samples)
    tokenized_prompts = get_tokenized_prompts(eval_df, lang, "XNLI")
    outputs = generate_texts_batch(model, tokenizer, tokenized_prompts, max_new_tokens=10, batch_size=batch_size, desc=f"XNLI-{lang}")

    rows, correct = [], 0
    for (_, row), output in zip(eval_df.iterrows(), outputs):
        pred = extract_xnli_label(output)
        gold = row["gold"]
        is_correct = pred == gold
        correct += int(is_correct)
        rows.append({
            "premise": row["premise"],
            "hypothesis": row["hypothesis"],
            "raw_output": output,
            "pred": pred,
            "gold": gold,
            "correct": is_correct
        })
    return correct / len(rows) if rows else 0.0, pd.DataFrame(rows)

# ----------------------------
# CSQA
# ----------------------------
# Prompts, prompt tokens, and gold normalization are created in preprocessing.
CHOICE_RE = re.compile(r"\b([ABCDE])\b")
ANSWER_CHOICE_RE = re.compile(r"ANSWER\s*[:\-]?\s*([ABCDE])")

def extract_choice(text: str):
    text = text.strip().upper()
    match = CHOICE_RE.search(text)
    if match:
        return match.group(1)
    match = ANSWER_CHOICE_RE.search(text)
    if match:
        return match.group(1)
    return None

def evaluate_csqa(model, tokenizer, df, lang: str, n_samples=None, batch_size: int = 32):
    eval_df = df if n_samples is None else df.head(n_samples)
    tokenized_prompts = get_tokenized_prompts(eval_df, lang, "CSQA")
    outputs = generate_texts_batch(model, tokenizer, tokenized_prompts, max_new_tokens=10, batch_size=batch_size, desc=f"CSQA-{lang}")

    rows, correct = [], 0
    for (_, row), output in zip(eval_df.iterrows(), outputs):
        pred = extract_choice(output)
        gold = row["gold"]
        is_correct = pred == gold
        correct += int(is_correct)
        rows.append({
            "stem": row["stem"],
            "raw_output": output,
            "pred": pred,
            "gold": gold,
            "correct": is_correct
        })
    return correct / len(rows) if rows else 0.0, pd.DataFrame(rows)

# ----------------------------
# Main suites
# ----------------------------
def save_results(results, summary_path, full_path):
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump({"summary": results["summary"]}, f, ensure_ascii=False, indent=2)

    json_results = {
        k: (v.to_dict(orient="records") if isinstance(v, pd.DataFrame) else v)
        for k, v in results.items()
    }
    with open(full_path, "w", encoding="utf-8") as f:
        json.dump(json_results, f, ensure_ascii=False, indent=2)

def main_english(model, tokenizer, datasets, batch_size=32, n_samples=None):
    summary = {}

    summary["mgsm_zh_acc"], res_mgsm_zh = evaluate_mgsm(model, tokenizer, datasets["mgsm_zh"], lang="en", n_samples=n_samples, batch_size=batch_size)
    summary["mgsm_es_acc"], res_mgsm_es = evaluate_mgsm(model, tokenizer, datasets["mgsm_es"], lang="en", n_samples=n_samples, batch_size=batch_size)
    summary["mgsm_ur_acc"], res_mgsm_ur = evaluate_mgsm(model, tokenizer, datasets["mgsm_ur"], lang="en", n_samples=n_samples, batch_size=batch_size)
    print("MGSM Done - EN")

    summary["xnli_zh_acc"], res_xnli_zh = evaluate_xnli(model, tokenizer, datasets["xnli_zh"], lang="en", n_samples=n_samples, batch_size=batch_size)
    summary["xnli_es_acc"], res_xnli_es = evaluate_xnli(model, tokenizer, datasets["xnli_es"], lang="en", n_samples=n_samples, batch_size=batch_size)
    summary["xnli_ur_acc"], res_xnli_ur = evaluate_xnli(model, tokenizer, datasets["xnli_ur"], lang="en", n_samples=n_samples, batch_size=batch_size)
    print("XNLI Done - EN")

    summary["csqa_zh_acc"], res_csqa_zh = evaluate_csqa(model, tokenizer, datasets["csqa_zh"], lang="en", n_samples=n_samples, batch_size=batch_size)
    summary["csqa_es_acc"], res_csqa_es = evaluate_csqa(model, tokenizer, datasets["csqa_es"], lang="en", n_samples=n_samples, batch_size=batch_size)
    summary["csqa_ur_acc"], res_csqa_ur = evaluate_csqa(model, tokenizer, datasets["csqa_ur"], lang="en", n_samples=n_samples, batch_size=batch_size)
    print("CSQA Done - EN")

    return {
        "summary": summary,
        "mgsm_zh": res_mgsm_zh, "mgsm_es": res_mgsm_es, "mgsm_ur": res_mgsm_ur,
        "xnli_zh": res_xnli_zh, "xnli_es": res_xnli_es, "xnli_ur": res_xnli_ur,
        "csqa_zh": res_csqa_zh, "csqa_es": res_csqa_es, "csqa_ur": res_csqa_ur,
    }

def main_language(model, tokenizer, datasets, batch_size=32, n_samples=None):
    summary = {}

    summary["mgsm_zh_acc"], res_mgsm_zh = evaluate_mgsm(model, tokenizer, datasets["mgsm_zh"], lang="zh", n_samples=n_samples, batch_size=batch_size)
    summary["mgsm_es_acc"], res_mgsm_es = evaluate_mgsm(model, tokenizer, datasets["mgsm_es"], lang="es", n_samples=n_samples, batch_size=batch_size)
    summary["mgsm_ur_acc"], res_mgsm_ur = evaluate_mgsm(model, tokenizer, datasets["mgsm_ur"], lang="ur", n_samples=n_samples, batch_size=batch_size)
    print("MGSM Done - Lang")

    summary["xnli_zh_acc"], res_xnli_zh = evaluate_xnli(model, tokenizer, datasets["xnli_zh"], lang="zh", n_samples=n_samples, batch_size=batch_size)
    summary["xnli_es_acc"], res_xnli_es = evaluate_xnli(model, tokenizer, datasets["xnli_es"], lang="es", n_samples=n_samples, batch_size=batch_size)
    summary["xnli_ur_acc"], res_xnli_ur = evaluate_xnli(model, tokenizer, datasets["xnli_ur"], lang="ur", n_samples=n_samples, batch_size=batch_size)
    print("XNLI Done - Lang")

    summary["csqa_zh_acc"], res_csqa_zh = evaluate_csqa(model, tokenizer, datasets["csqa_zh"], lang="zh", n_samples=n_samples, batch_size=batch_size)
    summary["csqa_es_acc"], res_csqa_es = evaluate_csqa(model, tokenizer, datasets["csqa_es"], lang="es", n_samples=n_samples, batch_size=batch_size)
    summary["csqa_ur_acc"], res_csqa_ur = evaluate_csqa(model, tokenizer, datasets["csqa_ur"], lang="ur", n_samples=n_samples, batch_size=batch_size)
    print("CSQA Done - Lang")

    return {
        "summary": summary,
        "mgsm_zh": res_mgsm_zh, "mgsm_es": res_mgsm_es, "mgsm_ur": res_mgsm_ur,
        "xnli_zh": res_xnli_zh, "xnli_es": res_xnli_es, "xnli_ur": res_xnli_ur,
        "csqa_zh": res_csqa_zh, "csqa_es": res_csqa_es, "csqa_ur": res_csqa_ur,
    }

def main():
    start = time.time()
    print(f"Starting mode={args.mode}")

    model, tokenizer = load_model()
    datasets = load_all_datasets()

    if args.mode == "english":
        results = main_english(model, tokenizer, datasets, batch_size=args.batch_size, n_samples=args.n_samples)
        summary_path = f"/kaggle/working/{cond}_results_summary_english.json"
        full_path    = f"/kaggle/working/{cond}_results_english.json"
    else:
        results = main_language(model, tokenizer, datasets, batch_size=args.batch_size, n_samples=args.n_samples)
        summary_path = f"/kaggle/working/{cond}_results_summary_language.json"
        full_path    = f"/kaggle/working/{cond}_results_language.json"

    save_results(results, summary_path, full_path)

    print(f"Saved: {summary_path}")
    print(f"Saved: {full_path}")
    print(f"Elapsed seconds: {time.time() - start:.2f}")

if __name__ == "__main__":
    main()

Writing /kaggle/working/run_eval_single.py


In [3]:
!CUDA_VISIBLE_DEVICES=0 python /kaggle/working/run_eval_single.py --mode english --batch_size 32 > /kaggle/working/english.log 2>&1 & \
 CUDA_VISIBLE_DEVICES=1 python /kaggle/working/run_eval_single.py --mode language --batch_size 32 > /kaggle/working/language.log 2>&1 & \
 wait

^C
